# Efektywne programowanie w języku Python
## Django — część 2
Formularze, logowanie i prywatne notatki

### Kontynuacja części pierwszej
Znamy model, migracje, admin, listę i szczegóły notatki.
Teraz dodajemy formularze użytkownika i kontrolę dostępu do obiektów.

Blok opcjonalny. Python 3.10, Django 5.2 LTS. Przykłady dotyczą tej wersji, nie historycznego Django 1.11.

In [ ]:
import sys
import django
print("Python:", sys.version.split()[0], "Django:", django.get_version())
assert django.VERSION[:2] == (5, 2)

### Co zbudujemy?
- Logowanie i wylogowanie.
- Listę i szczegóły tylko własnych notatek.
- Tworzenie, edycję i potwierdzanie usunięcia.
- Formularz kontaktowy jako demonstrację walidacji.
- Testy poprawnych operacji i prób obejścia uprawnień.

Kontakt nie wysyła e-maila ani nie zapisuje zgłoszeń.

### Gotowa demonstracja i czysta baza
[Instrukcja](part10_django2_files/README.md), [projekt](part10_django2_files/demo/).
```text
python -m pip install -r part10_django2_files/requirements.txt
cd part10_django2_files/demo
python manage.py migrate
python manage.py createsuperuser
python manage.py runserver 127.0.0.1:8000
```
Korzystamy z osobnej bazy. Nie kopiuj bazy części pierwszej: przypisanie właścicieli istniejącym notatkom wymaga planu migracji danych.

## 1. Żądanie i podział odpowiedzialności
Przeglądarka → middleware → routing → widok → model/formularz → szablon → odpowiedź.

Model opisuje dane. Formularz wiąże i waliduje wejście. Widok decyduje o operacji i dostępie. Szablon prezentuje wynik.
Terminologia MTV opisuje role, nie wymusza dokładnego odwzorowania nazw MVC.

### Funkcja widoku też jest dobrym narzędziem
Widok przyjmuje `request` i zwraca odpowiedź HTTP.
`render()` łączy szablon, kontekst i odpowiedź.

Nie sklejamy HTML-a z danymi użytkownika. `timezone.now()` pasuje do świadomych stref czasowych przy `USE_TZ=True`.

### Prosty widok funkcyjny
Plik: `notes/views.py`

```python
def about(request):
    return render(request, "notes/about.html", {"now": timezone.now()})
```

### Szablon widoku funkcyjnego
Plik `notes/templates/notes/about.html`:
```html
{% extends "notes/base.html" %}
{% block content %}
  <h1>O aplikacji</h1>
  <p>Aktualny czas: {{ now|date:"d.m.Y H:i" }}</p>
{% endblock %}
```
`{{ ... }}` wstawia wartość, `{% ... %}` wykonuje tag, `{# ... #}` jest komentarzem szablonu.

### Routing bez historycznego `patterns()` i `url()`
```python
from django.urls import path
from . import views

app_name = "notes"
urlpatterns = [
    path("about/", views.about, name="about"),
]
```
Nazwy URL-i umożliwiają `reverse()` oraz `{% url 'notes:about' %}`. Nie wpisujemy fikcyjnego zewnętrznego adresu jako instrukcji do lokalnego projektu.

## 2. Relacja notatki do użytkownika
Do `Note` dodajemy:
```python
from django.conf import settings

owner = models.ForeignKey(
    settings.AUTH_USER_MODEL,
    on_delete=models.CASCADE,
    related_name="notes",
    verbose_name="właściciel",
)
```
Używamy konfigurowalnego modelu użytkownika, zamiast importować na stałe konkretną klasę `User`.

### Co oznacza relacja?
Użytkownik ma wiele notatek, notatka jednego właściciela.
`user.notes.all()` korzysta z `related_name`.

`CASCADE` w operacji usuwania przez Django oznacza usunięcie powiązanych notatek po usunięciu użytkownika. Wymagania retencji mogą uzasadniać inny wybór, np. `PROTECT`.
`null` dotyczy bazy, `blank` walidacji formularzy/modelu — nie są zamienne.

### Inne relacje — świadomy wybór
- `ForeignKey`: wiele rekordów do jednego, np. notatki autora.
- `OneToOneField`: co najwyżej jeden powiązany rekord, np. profil.
- `ManyToManyField`: wiele do wielu, np. tagi notatek.

Nie przekazujemy niezdefiniowanych `Poll`, `Site` czy `Place` jako działającego kodu. Najpierw projektujemy konkretny model i jego kontrakt.

### Migracja schematu a migracja istniejących danych
```text
python manage.py makemigrations notes
python manage.py migrate
```
W demonstracji `0002_note_owner.py` już istnieje i działa na nowej bazie.

W bazie z notatkami bez właściciela nie zgaduj jednego domyślnego użytkownika. Zaplanuj przypisanie danych, np. etap pola opcjonalnego, migrację danych i dopiero wymagane pole.

### ORM: filtruj według celu
Przykład w `python manage.py shell`:
```python
from django.contrib.auth import get_user_model
from notes.models import Note

user = get_user_model().objects.get(username="alice")
notes = Note.objects.filter(owner=user, title__icontains="plan")
```
QuerySet jest leniwy. Ustawienie filtra właściciela w shellu nie zabezpiecza samo widoków — muszą stosować własne filtrowanie.

## 3. Formularz to także walidacja
GET wyświetla niezwiązany formularz. POST wiąże dane, także gdy przesłany słownik jest pusty.
`is_valid()` udostępnia oczyszczone dane w `cleaned_data` lub błędy w `errors`.

Nie korzystamy z `request.POST or None`: pusty POST powinien pozostać formularzem związanym i pokazać błędy.

### Formularz niezależny od modelu
Plik: `notes/forms.py`

```python
class ContactForm(forms.Form):
    subject = forms.CharField(label="Temat", max_length=100)
    message = forms.CharField(label="Wiadomość", widget=forms.Textarea)
    sender = forms.EmailField(label="E-mail")
    cc_myself = forms.BooleanField(label="Kopia dla mnie", required=False)
```

### Checkbox i oczyszczone dane
`BooleanField(required=False)` daje wartość logiczną także wtedy, gdy checkbox nie jest zaznaczony.

Nie zakładaj, że tekst z `request.POST` ma już właściwy typ. Korzystaj z `cleaned_data` po udanej walidacji. Dodatkowe reguły pojedynczego pola zapisujemy w `clean_<pole>()`, a reguły wielu pól w `clean()`.

### Obsługa GET i POST formularza
Plik: `notes/views.py`

```python
@login_required
def contact(request):
    if request.method == "POST":
        form = ContactForm(request.POST)
        if form.is_valid():
            # Demo walidacji: bez wysyłki e-mail i bez zapisu danych.
            return redirect("notes:contact-thanks")
    else:
        form = ContactForm()
    return render(request, "notes/contact.html", {"form": form})
```

### POST → redirect → GET
Po poprawnym POST przekierowujemy do nazwanej strony podziękowania. Odświeżenie tej strony nie powtarza poprzedniego POST.

To wzorzec PRG. Samo przekierowanie nie gwarantuje jednorazowego wykonania operacji przy ponowieniu żądania.
Nasz kontakt demonstruje walidację i przepływ — nie implementuje wysyłki wiadomości.

### Szablon formularza kontaktowego
Plik: `notes/templates/notes/contact.html`

```html
{% extends "notes/base.html" %}
{% block content %}<h1>Kontakt — demonstracja formularza</h1>
<p>Dane służą wyłącznie walidacji; nie wysyłamy e-maila ani nie zapisujemy zgłoszenia.</p>
<form method="post">{% csrf_token %}{{ form.as_p }}<button>Sprawdź formularz</button></form>
{% endblock %}
```

### CSRF i logowanie to różne zabezpieczenia
`{% csrf_token %}` w formularzu POST współpracuje z middleware CSRF. Brak prawidłowego tokenu może dać 403.

Token nie zastępuje logowania ani kontroli właściciela obiektu. Zalogowany użytkownik nadal może próbować podać cudzy identyfikator.
Nie wyłączamy CSRF, aby usunąć komunikat błędu.

## 4. `ModelForm` z jawną listą pól
W formularzu notatki edytujemy tylko `title` i `body`. Właściciela ustala serwer.

`fields = "__all__"` mogłoby przypadkowo udostępnić nowe pole po zmianie modelu. Jawna lista określa kontrakt edycji.

### Formularz notatki i walidacja tytułu
Plik: `notes/forms.py`

```python
from django import forms
from .models import Note

class NoteForm(forms.ModelForm):
    class Meta:
        model = Note
        fields = ["title", "body"]

    def clean_title(self):
        title = self.cleaned_data["title"].strip()
        if len(title) < 3:
            raise forms.ValidationError("Tytuł musi mieć co najmniej 3 znaki.")
        return title

```

### Gdzie odbywa się walidacja?
`ModelForm` łączy walidację pól formularza i modelu dla uwzględnionych pól.
`clean_title()` usuwa otaczające odstępy i odrzuca zbyt krótki tytuł.

`model.save()` nie uruchamia automatycznie `full_clean()`. Ograniczenia bazy oraz uprawnienia muszą być zaprojektowane osobno.

## 5. Uwierzytelnianie i sesja
Do URL-i projektu dodajemy:
```python
path("accounts/", include("django.contrib.auth.urls")),
```
W ustawieniach:
```python
LOGIN_URL = "login"
LOGIN_REDIRECT_URL = "notes:note-list"
LOGOUT_REDIRECT_URL = "login"
```
Django zapewnia widoki auth, ale własny szablon logowania trzeba przygotować.

### Szablon logowania
Plik: `notes/templates/registration/login.html`

```html
{% extends "notes/base.html" %}
{% block content %}
<h1>Zaloguj się</h1>
<form method="post">{% csrf_token %}
{{ form.as_p }}
<input type="hidden" name="next" value="{{ next }}">
<button type="submit">Zaloguj</button></form>
{% endblock %}
```

### Wylogowanie przez POST
W szablonie dla zalogowanego użytkownika:
```html
<form method="post" action="{% url 'logout' %}">
  {% csrf_token %}
  <button>Wyloguj</button>
</form>
```
W Django 5.2 GET nie wylogowuje użytkownika. Nie zastępuj formularza zwykłym linkiem.
Django sprawdza dozwolony cel przekierowania logowania; nie buduj własnego niekontrolowanego `redirect(request.GET['next'])`.

### Logowanie nie oznacza dostępu do każdego obiektu
`LoginRequiredMixin` wymaga użytkownika, ale nie filtruje notatek według właściciela.

Wspólny mixin zawęża QuerySet. Dzięki temu cudzy identyfikator daje 404 także w widoku szczegółu, edycji i usunięcia.
Ukrycie przycisku w HTML nie wystarcza.

### Zawężony QuerySet dla widoków
Plik: `notes/views.py`

```python
class OwnedNotesMixin(LoginRequiredMixin):
    model = Note
    def get_queryset(self):
        return super().get_queryset().filter(owner=self.request.user)

class NoteListView(OwnedNotesMixin, ListView):
    context_object_name = "notes"
    paginate_by = 5

class NoteDetailView(OwnedNotesMixin, DetailView):
    context_object_name = "note"
```

### Kolejność mixinów
`OwnedNotesMixin` dziedziczy po `LoginRequiredMixin` i występuje przed klasą widoku generycznego.
`super().get_queryset()` współpracuje z MRO znanym z `part07`.

Ochrona obowiązuje dla naszych widoków. Admin ma własną kontrolę uprawnień; standardowo nie ograniczy automatycznie listy do właściciela.

## 6. Tworzenie notatki
Użytkownik przesyła tytuł i treść. Serwer przypisuje właściciela z sesji.

Nigdy nie ufamy polu `owner` otrzymanemu z formularza lub ręcznie przygotowanego POST. Dodatkowe pola nieuwzględnione w `ModelForm` nie są zapisywane przez ten formularz.

### CreateView i właściciel
Plik: `notes/views.py`

```python
class NoteCreateView(LoginRequiredMixin, CreateView):
    model = Note
    form_class = NoteForm
    success_url = reverse_lazy("notes:note-list")
    def form_valid(self, form):
        form.instance.owner = self.request.user
        return super().form_valid(form)
```

### Wspólny szablon tworzenia i edycji
Plik: `notes/templates/notes/note_form.html`

```html
{% extends "notes/base.html" %}
{% block content %}
<h1>{% if object %}Edytuj notatkę{% else %}Dodaj notatkę{% endif %}</h1>
<form method="post">
  {% csrf_token %}
  {{ form.as_p }}
  <button type="submit">Zapisz</button>
</form>
{% endblock %}
```

### Edycja notatki
```python
class NoteUpdateView(OwnedNotesMixin, UpdateView):
    form_class = NoteForm
    success_url = reverse_lazy("notes:note-list")
```
`UpdateView` pobiera obiekt ze zwężonego QuerySet. GET wyświetla dane, poprawny POST zapisuje zmiany.
`reverse_lazy()` opóźnia rozwiązywanie URL-a, co pasuje do atrybutów klasy definiowanej podczas importu.

### Usunięcie wymaga potwierdzenia
```python
class NoteDeleteView(OwnedNotesMixin, DeleteView):
    success_url = reverse_lazy("notes:note-list")
```
GET pokazuje pytanie i niczego nie usuwa. Dopiero poprawny POST usuwa obiekt.
Filtr właściciela musi działać dla obu metod.

### Potwierdzenie usunięcia
Plik: `notes/templates/notes/note_confirm_delete.html`

```html
{% extends "notes/base.html" %}
{% block content %}
<h1>Usunąć „{{ object.title }}”?</h1>
<form method="post">{% csrf_token %}<button type="submit">Potwierdź usunięcie</button></form>
<a href="{% url 'notes:note-detail' object.pk %}">Anuluj</a>
{% endblock %}
```

### Ścieżki operacji na notatkach
```python
path("", views.NoteListView.as_view(), name="note-list"),
path("new/", views.NoteCreateView.as_view(), name="note-create"),
path("<int:pk>/", views.NoteDetailView.as_view(), name="note-detail"),
path("<int:pk>/edit/", views.NoteUpdateView.as_view(), name="note-update"),
path("<int:pk>/delete/", views.NoteDeleteView.as_view(), name="note-delete"),
```
Wszystkie znajdują się w namespace `notes`. Funkcje widoków przekazujemy bez wywołania, klasy przez `as_view()`.

### Szablon szczegółu: przyciski to nawigacja
```html
<a href="{% url 'notes:note-update' note.pk %}">Edytuj</a>
<a href="{% url 'notes:note-delete' note.pk %}">Usuń</a>
```
Link „Usuń” prowadzi do potwierdzenia; samo kliknięcie nie usuwa rekordu.
Autoescaping treści z pierwszej części nadal obowiązuje. Nie używamy `safe` dla niesprawdzonej treści.

## 7. Sprawdź także niedozwolone operacje
Sukces zapisu to tylko część testów. Sprawdzamy też:
- niezalogowanego klienta;
- cudzy obiekt i podmieniony `owner`;
- pusty lub niepoprawny formularz;
- brak CSRF;
- GET usuwania i POST usuwania;
- HTML w treści oraz wylogowanie.

### Test ochrony cudzych notatek
Plik: `notes/tests.py`

```python
def test_other_object_get_denied(self):
    for suffix in ["", "edit/", "delete/"]:
        url = f"/notes/{self.other.pk}/{suffix}"
        self.assertEqual(self.client.get(url).status_code, 404)

# self.other jest notatką innego użytkownika niż zalogowany klient.
```

### Testy a ochrona CSRF
Domyślny klient testowy Django upraszcza obsługę CSRF. Do osobnego testu ochrony używamy:
```python
client = Client(enforce_csrf_checks=True)
```
`force_login()` pozwala skupić test obiektu na uprawnieniach. Osobny test sprawdza rzeczywisty przepływ logowania.
Nie uznajemy samego statusu 302 za dowód poprawnej autoryzacji.

### Automatyczna kontrola demonstracji
Z katalogu repozytorium:
```text
python part10_django2_files/verify_demo.py
```
Projekt kopiowany jest do katalogu tymczasowego. Kontrola obejmuje migracje, system checks i 16 testów.
Robocza baza zajęć nie jest zmieniana.

In [ ]:
from pathlib import Path
import os
import subprocess

completed = subprocess.run(
    [sys.executable, str(Path("part10_django2_files/verify_demo.py").resolve())],
    check=True, capture_output=True, text=True, encoding="utf-8",
    env={**os.environ, "PYTHONIOENCODING": "utf-8"}, timeout=120)
print(completed.stdout)

### Demonstracja na żywo — dwa konta
1. Zaloguj się jako A i dodaj notatkę.
2. Otwórz edycję, wyślij zbyt krótki tytuł, popraw i zapisz.
3. Zaloguj się jako B: lista nie pokazuje notatki A.
4. Wpisz ręcznie adres szczegółu i edycji A — oczekuj 404.
5. Wróć do A, otwórz usuwanie, anuluj, potem potwierdź.

Admin służy tu do przygotowania kont, nie do demonstracji filtrowania naszych widoków.

### Rozszerzenie: koszt ORM i `select_related`
Gdy szablon wyświetla właściciela każdej notatki, pobieranie relacji osobnymi zapytaniami może dać problem N+1.
Dla ForeignKey użyteczne bywa `select_related("owner")`, dla kolekcji `prefetch_related()`.

W naszej liście nie odczytujemy właściciela każdego wiersza. Nie dodajemy optymalizacji bez potrzeby; najpierw sprawdzamy zapytania.

### Rozszerzenie: PostgreSQL
Nowy backend to `django.db.backends.postgresql`. Konfigurację loginu i hasła pobieraj ze środowiska, nie ze slajdu.

Sterownik i serwer bazy wymagają osobnego przygotowania oraz weryfikacji zgodności. Ten projekt jest sprawdzany na SQLite; nie twierdzimy, że wykonano demonstrację PostgreSQL.

### Rozszerzenie: zapis wielu zmian
Dla kilku powiązanych operacji można użyć `transaction.atomic()`.
Nie zastępuje to walidacji, autoryzacji ani rozwiązania wszystkich wyścigów współbieżnych żądań.

Wysyłka e-maila to dodatkowy efekt poza bazą — rozważ uruchomienie po udanej transakcji przez `transaction.on_commit()`. W tym wykładzie nie wysyłamy wiadomości.

### Eksperymenty: uprawnienia i spójność
Przewidź wynik trzech prób: cudzy identyfikator, dwa zwiększenia licznika i błąd między zapisami.

[Skrypt eksperymentów](part10_django_files/experiments.py) używa izolowanych mini-modeli na SQLite w pamięci. Logikę izolacji prawdziwych użytkowników sprawdzają dodatkowo testy aplikacji z części drugiej.

### Zmiana identyfikatora: znajdź lukę
Antywzorzec w izolowanym eksperymencie:
```python
Entry.objects.get(pk=entry_id)
```
Brakuje sprawdzenia właściciela. Samo logowanie i ukrycie linku nie naprawia tego błędu.
```python
get_object_or_404(
    Entry.objects.filter(owner=current_user), pk=entry_id)
```
W aplikacji `OwnedNotesMixin` stosuje tę zasadę w szczegółach, edycji i usuwaniu. Nie dodajemy celowo niezabezpieczonego URL-a do projektu.

### Dwa odczyty, jedna utracona aktualizacja
```python
request_a = Entry.objects.get(pk=pk)  # views == 0
request_b = Entry.objects.get(pk=pk)  # views == 0
request_a.views += 1
request_a.save(update_fields=["views"])
request_b.views += 1
request_b.save(update_fields=["views"])
```
Wynik: 1. Wymuszamy kolejność odczyt–odczyt–zapis–zapis, modelując przeplot dwóch żądań bez losowości harmonogramu wątków.

### Zwiększanie po stronie bazy przez `F`
```python
from django.db.models import F
Entry.objects.filter(pk=pk).update(views=F("views") + 1)
```
Baza zwiększa aktualną wartość w jednym SQL. Dwa takie wywołania dają 2; po odczycie używamy `refresh_from_db()`.

To rozwiązanie dla tej aktualizacji, nie uniwersalna ochrona wieloetapowych operacji. `update()` nie wywołuje `save()` ani jego sygnałów.

### Błąd między dwoma zapisami
```python
try:
    with transaction.atomic():
        Entry.objects.create(owner=alice, title="Pierwszy zapis")
        raise RuntimeError("Błąd przed drugim zapisem")
except RuntimeError:
    pass
```
Wyjątek wychodzi z `atomic`, więc pierwszy zapis jest wycofany. Bez transakcji pozostaje zatwierdzony przez autocommit.
Wyjątek obsługujemy poza blokiem transakcji.

In [ ]:
experiment = Path("part10_django_files/experiments.py").resolve()
result = subprocess.run(
    [sys.executable, str(experiment), "--part", "2"], check=True,
    capture_output=True, text=True, encoding="utf-8",
    env={**os.environ, "PYTHONIOENCODING": "utf-8"}, timeout=60)
print(result.stdout)

### Czego transakcja nie cofa?
`atomic()` dotyczy operacji bazy w danej transakcji. Nie cofnie e-maila, zapisu do pliku ani mutacji Pythonowego obiektu.

Dla akcji po udanym zatwierdzeniu użyj `transaction.on_commit()`, gdy odpowiada to wymaganiom. Nie traktuj tego jako gwarancji dostarczenia wiadomości.
Eksperyment na SQLite pokazuje kontrolowany przeplot i rollback, nie test obciążeniowy współbieżności produkcyjnej bazy.

## Ćwiczenia i pytania
1. Dlaczego `owner` nie jest polem `NoteForm`?
2. Czy samo logowanie blokuje odczyt cudzej notatki?
3. Dlaczego GET usuwania nie powinien usuwać?
4. Co dzieje się przy pustym POST?
5. Czy formularz kontaktowy w tej demonstracji wysyła e-mail?

### Odpowiedzi
1. Właściciela ustala serwer z sesji, nie dane klienta.
2. Nie — obiekty trzeba filtrować lub sprawdzać ich uprawnienia.
3. GET służy odczytowi; usunięcie wymaga jawnej operacji.
4. Formularz jest związany i pokazuje błędy wymaganych pól.
5. Nie — demonstruje walidację i przekierowanie PRG.

### Zadanie rozszerzające
Dodaj znacznik „ważna” do notatki: pole modelu, migrację, jawną listę pól formularza, prezentację i testy.

Następnie dodaj wyszukiwanie we własnych notatkach. Zachowaj filtr właściciela, także przy zapytaniu pustym i przy paginacji.
Nie zmieniaj kontroli dostępu, aby uprościć demonstrację.

### Self reading — formularze i widoki
- [Formularze Django 5.2](https://docs.djangoproject.com/en/5.2/topics/forms/)
- [ModelForm](https://docs.djangoproject.com/en/5.2/topics/forms/modelforms/)
- [Widoki generyczne edycji](https://docs.djangoproject.com/en/5.2/topics/class-based-views/generic-editing/)
- [Uwierzytelnianie i widoki auth](https://docs.djangoproject.com/en/5.2/topics/auth/default/)
- [Ochrona CSRF](https://docs.djangoproject.com/en/5.2/howto/csrf/)

Materiały są opcjonalnym rozszerzeniem kursu, bez dawnych obowiązkowych zadań przypisanych osobom.

### Self reading — modele i testowanie
- [Pola modeli i relacje](https://docs.djangoproject.com/en/5.2/ref/models/fields/)
- [Zapytania ORM](https://docs.djangoproject.com/en/5.2/topics/db/queries/)
- [Testy Django](https://docs.djangoproject.com/en/5.2/topics/testing/overview/)
- [Transakcje](https://docs.djangoproject.com/en/5.2/topics/db/transactions/)

Demo części 2 ma osobną bazę i kod. Część 1 pozostaje punktem odniesienia dla publicznej aplikacji odczytu.